In [ ]:
# ============================================
# 🇰🇷 한국어 질의응답기 체험 (Gradio)
# ============================================
# 아래 버전은 예제 동작을 확인한 버전입니다. 지우거나 바꾸지 마세요.
!pip -q install transformers==5.15.0 gradio==6.24.0

from transformers import AutoTokenizer, AutoModelForQuestionAnswering
import gradio as gr
import traceback
# 다국어 질의응답 모델 (한국어 지원)
model_name = "deepset/xlm-roberta-base-squad2"
qa_tokenizer = AutoTokenizer.from_pretrained(model_name)
qa_model = AutoModelForQuestionAnswering.from_pretrained(model_name)

# Q&A 함수
def answer_question_ko(context, question):
    try:
        inputs = qa_tokenizer(question, context, return_tensors="pt", truncation="only_second", max_length=384)
        out = qa_model(**inputs)

        # 지문의 단어(토큰)마다 매겨진 '시작·끝 확률' 중 값이 가장 큰 단어의 번호를 찾는다
        start_p = out.start_logits.softmax(-1)[0]
        end_p = out.end_logits.softmax(-1)[0]
        start, end = int(start_p.argmax()), int(end_p.argmax())

        answer = qa_tokenizer.decode(inputs["input_ids"][0][start:end + 1],
                                  skip_special_tokens=True).strip()
        if not answer:
            return "❓ 지문에서 답을 찾지 못했습니다."
        return f"✅ 정답: {answer}\n📊 확신도: {(start_p[start] * end_p[end]).item():.2%}"
    except Exception:
        return "⚠️ 오류 발생:\n" + traceback.format_exc()
# Gradio 인터페이스
gr.Interface(
    fn=answer_question_ko,
    inputs=[
        gr.Textbox(lines=6, label="📚 지문 (Context)", value="기상이변 또는 극한기후 또는 이상기상은 평년 기후의 수준을 크게 벗어난 비일상적, 비정상적, 극단적 기상현상을 의미하며, 보통 30년을 기준으로 삼는다. 기상이변의 원인은 여러 가지이며, 지구 온난화, 엘니뇨, 북극진동, 제트기류 등이다. 하지만 엘니뇨와 라니냐는 수년에 한 번씩 찾아오는 현상으로 전제해 본다면 기상이변은 지구온난화에 기인한다고 할 수 있다."),
        gr.Textbox(lines=1, label="❓ 질문 (Question)", value="기상이변의 핵심 원인은?"),
    ],
    outputs=gr.Textbox(label="🧠 AI의 대답"),
    title="🇰🇷 한국어 질의응답기",
    description="한국어 지문을 바탕으로 질문을 하면, AI가 정답을 찾아줍니다!",
).launch()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b84b1806292d5a1826.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# ============================================
# 🇰🇷 한국어 요약기 체험 (KoBART / Gradio)
# ============================================
# 아래 버전은 예제 동작을 확인한 버전입니다. 지우거나 바꾸지 마세요.
!pip -q install transformers==5.15.0 gradio==6.24.0 sentencepiece==0.2.2

import gradio as gr
from transformers import PreTrainedTokenizerFast, BartForConditionalGeneration

# 1. 모델과 토크나이저 로드 (공개 모델 사용)
model_name = "digit82/kobart-summarization"
sum_tokenizer = PreTrainedTokenizerFast.from_pretrained(model_name)
sum_model = BartForConditionalGeneration.from_pretrained(model_name)

# 2. 요약 함수
def summarize_ko(text):
    inputs = sum_tokenizer(text, return_tensors="pt", max_length=1024, truncation=True)
    summary_ids = sum_model.generate(
        **inputs,
        max_new_tokens=128,    # 새로 만들 최대 길이
        min_new_tokens=10,     # 너무 짧게 끝내지 않도록
    )
    return sum_tokenizer.decode(summary_ids[0], skip_special_tokens=True)

# Gradio 인터페이스
gr.Interface(
    fn=summarize_ko,
    inputs=gr.Textbox(lines=8, label="📚 요약할 한국어 문장을 입력하세요", value="""
엘니뇨(→어린 남자아이)는 엘니뇨 남방진동의 따듯한 단계로, 남아메리카 태평양 해안 등 동태평양의 해수가 따듯해지는 현상이다. 엘니뇨 남방진동은 동태평양의 해수면 온도가 따듯한 단계와 차가운 단계를 왕복하는 과정이다.
엘니뇨가 발생하면 서태평양에는 고기압, 동태평양에는 저기압이 형성된다. 보통 엘니뇨는 4년 가량 지속되는데, 기록에 따르면 엘니뇨의 주기는 2~7년 사이이다. 엘니뇨에 반대되는 단계는 라니냐(→어린 여자아이)라고 부르며, 서태평양에 저기압, 동태평양에 고기압이 생긴다. 엘니뇨와 라니냐를 아우르는 엘니뇨 남방진동은 전 세계적으로 기온과 강수량 변화를 일으킨다. 특히 태평양 연안에 위치한, 경제를 농업과 수산업에 의존하는 개발도상국이 가장 큰 영향을 받는다.
엘니뇨 시기 남아메리카 해안에서는 크리스마스 시기에 수온이 가장 높아지며, 엘니뇨를 가리키던 원래 용어인 엘니뇨데나비다드는 페루의 한 어부가 이 현상의 이름을 아기 예수에 빗대 붙인 것에서 유래하였다.
"""),
    outputs=gr.Textbox(label="📝 요약 결과"),
    title="🧠 한국어 텍스트 요약기 (KoBART)",
    description="긴 한국어 문장을 짧고 알기 쉽게 요약해주는 AI 요약 체험 도우미입니다.",
).launch()


Loading weights:   0%|          | 0/262 [00:00<?, ?it/s]

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://635c1143b50f07a06f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# ============================================
# 🌐 다국어 번역기 체험 (Gradio)
# ============================================
# 아래 버전은 예제 동작을 확인한 버전입니다. 지우거나 바꾸지 마세요.
!pip -q install transformers==5.15.0 gradio==6.24.0 sentencepiece==0.2.2

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import gradio as gr

model_id = "facebook/nllb-200-distilled-600M"
trans_tokenizer = AutoTokenizer.from_pretrained(model_id)
trans_model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

def translate_text(text, direction):
    if direction == "영어 → 한국어":
        trans_tokenizer.src_lang, target = "eng_Latn", "kor_Hang"
    else:
        trans_tokenizer.src_lang, target = "kor_Hang", "eng_Latn"

    inputs = trans_tokenizer(text, return_tensors="pt")
    ids = trans_model.generate(
        **inputs,
        forced_bos_token_id=trans_tokenizer.convert_tokens_to_ids(target),  # 목표 언어 지정
        max_new_tokens=256,
    )
    return trans_tokenizer.decode(ids[0], skip_special_tokens=True)

# Gradio UI 구성
gr.Interface(
    fn=translate_text,
    inputs=[
        gr.Textbox(lines=4, label="📝 번역할 문장", value="I want to learn artificial intelligence."),
        gr.Radio(["영어 → 한국어", "한국어 → 영어"], label="🔁 번역 방향", value="영어 → 한국어")
    ],
    outputs=gr.Textbox(label="🗣️ 번역 결과"),
    title="🌐 AI 다국어 번역기 (NLLB)",
    description="AI가 입력한 문장을 자동으로 번역해줍니다. 영어 ↔ 한국어 번역을 선택해 실험해보세요!",
).launch()


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a523576726d84ff735.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
